# NEXUS: XGBoost with constrained stable interactions

A CUDA XGBoost classifier restricts tree paths to stable or seasonal features. Matched unconstrained XGBoost and CatBoost runs are controls.

Enable a Kaggle GPU and attach the seven-file competition dataset. Run all cells in order. The final CSV and diagnostics are saved in `/kaggle/working`. The model is scratch-trained using only supplied data. The July–December 2024 fold has been repeatedly examined and is reported as a development diagnostic. No result is claimed until this notebook runs.


## 1. Competition feature implementation

Embedded from `optiml/features.py`; the notebook has no repository dependency.


In [ ]:
"""Target-independent joins and features. Raw CSV files are never modified."""
from pathlib import Path
import numpy as np
import pandas as pd

VERSION = 'strategy-v1'
KEYS = ['order_line_id', 'order_id', 'customer_id', 'product_id', 'store_id']
CATEGORY_TYPOS = {
    'electronicsaccessories': 'electronics_accessories', 'fashion_acessories': 'fashion_accessories',
    'food_beerages': 'food_beverages', 'furnture': 'furniture', 'garde_diy': 'garden_diy',
    'healthbeauty': 'health_beauty', 'home_ecor': 'home_decor', 'kitchendining': 'kitchen_dining',
    'office_upplies': 'office_supplies', 'sports_eisure': 'sports_leisure',
    'technlogy': 'technology', 'toys_ifts': 'toys_gifts',
}

def normalize(s):
    return s.astype('string').str.strip().str.lower().str.replace(r'[\s-]+', '_', regex=True)

def ratio(a, b):
    return a.div(b.where(b != 0)).replace([np.inf, -np.inf], np.nan)

def dimensions(data_dir):
    root = Path(data_dir)
    customer = pd.read_csv(root / 'customers.csv')
    customer['record_updated_at'] = pd.to_datetime(customer.record_updated_at, errors='raise')
    customer = customer.sort_values(['customer_id', 'record_updated_at'], kind='stable').drop_duplicates('customer_id', keep='last')
    # Supplied-snapshot policy for region; mutable loyalty and update time are excluded.
    customer = customer.drop(columns=['record_updated_at', 'loyalty_tier'])
    customer['region'] = normalize(customer.region)
    product = pd.read_csv(root / 'products.csv').drop(columns=['launch_date', 'review_count'])
    product['category'] = normalize(product.category).replace(CATEGORY_TYPOS)
    store = pd.read_csv(root / 'stores.csv').drop(columns=['seller_rating', 'manager_tenure_years'])
    store['region'] = normalize(store.region)
    calendar = pd.read_csv(root / 'calendar.csv')
    calendar['week_start'] = pd.to_datetime(calendar.week_start)
    result = []
    for frame, key, prefix in [(customer, 'customer_id', 'customer'), (product, 'product_id', 'product'), (store, 'store_id', 'store')]:
        if frame[key].duplicated().any():
            raise ValueError(f'Duplicate dimension key: {key}')
        result.append(frame.rename(columns={c: f'{prefix}_{c}' for c in frame if c != key}))
    result.append(calendar)
    return result

def build_features(raw, dims):
    df = raw.copy()
    if df.order_line_id.duplicated().any():
        raise ValueError('Duplicate order_line_id')
    original_ids = df.order_line_id.copy()
    ts = pd.to_datetime(df.order_ts, errors='raise')
    # Raw wall-clock timestamps: no unjustified timezone conversion.
    # Assign every order to the earliest line time so an order cannot straddle
    # a temporal fold when line timestamps differ.
    df['_order_time'] = ts.groupby(df.order_id).transform('min')
    df['_month'] = df['_order_time'].dt.strftime('%Y-%m')
    df['order_timestamp_span_hours'] = (df['_order_time'] - ts.groupby(df.order_id).transform('min')).dt.total_seconds() / 3600
    for dim, key in zip(dims[:3], ['customer_id', 'product_id', 'store_id']):
        if not df[key].isin(dim[key]).all():
            raise ValueError(f'Unmatched {key}')
        df = df.merge(dim, on=key, how='left', sort=False, validate='many_to_one')
    df['week_start'] = ts.dt.normalize() - pd.to_timedelta(ts.dt.dayofweek, unit='D')
    df = df.merge(dims[3], on='week_start', how='left', sort=False, validate='many_to_one', indicator='_calendar_join')
    if not df['_calendar_join'].eq('both').all():
        raise ValueError('Unmatched calendar week')
    if not df.order_line_id.reset_index(drop=True).equals(original_ids.reset_index(drop=True)):
        raise AssertionError('Join changed row order or count')
    df = df.drop(columns=['_calendar_join', 'week_start'])
    df['month_of_year'] = ts.dt.month
    df['day_of_week'] = ts.dt.dayofweek
    df['hour'] = ts.dt.hour
    df['weekend'] = (ts.dt.dayofweek >= 5).astype('int8')
    for field, out in [('customer_signup_date', 'customer_tenure_days'), ('store_opened_date', 'store_age_days')]:
        age = (ts - pd.to_datetime(df[field], errors='raise')).dt.total_seconds() / 86400
        df[out + '_invalid'] = (age < 0).astype('int8')
        df[out] = age.where(age >= 0)
        df = df.drop(columns=field)
    df['customer_store_region_match'] = (df.customer_region == df.store_region).astype('int8')
    df['line_gross_candidate'] = df.unit_price * df.quantity
    df['price_to_list_ratio'] = ratio(df.unit_price, df.product_list_price)
    df['amount_to_gross_ratio'] = ratio(df.order_value, df.line_gross_candidate)
    df['amount_gross_residual'] = df.order_value - df.line_gross_candidate
    df['freight_per_unit'] = ratio(df.freight_cost, df.quantity)
    df['freight_to_gross_ratio'] = ratio(df.freight_cost, df.line_gross_candidate)
    df['implied_discount_residual'] = (1 - df.price_to_list_ratio) - df.discount
    df['product_volume_cm3'] = df.product_length_cm * df.product_width_cm * df.product_height_cm
    df['product_density_kg_cm3'] = ratio(df.product_weight_kg, df.product_volume_cm3)
    df['line_weight_kg'] = df.product_weight_kg * df.quantity
    df['coupon_present'] = df.coupon_code.notna().astype('int8')
    for c in ['payment_method', 'ship_mode', 'discount', 'freight_cost', 'session_clicks', 'product_weight_kg', 'customer_age_band', 'store_size_sqm']:
        df[c + '_missing'] = df[c].isna().astype('int8')
    group = df.groupby('order_id', sort=False)
    df['basket_lines'] = group.order_line_id.transform('size')
    if not df.basket_lines.eq(df.items_in_order).all():
        raise ValueError('Incomplete orders or inconsistent items_in_order')
    df['basket_units'] = group.quantity.transform('sum')
    df['basket_gross_candidate'] = group.line_gross_candidate.transform('sum')
    df['basket_categories'] = group.product_category.transform('nunique')
    df['basket_products'] = group.product_id.transform('nunique')
    df['basket_stores'] = group.store_id.transform('nunique')
    df['basket_mean_discount'] = group.discount.transform('mean')
    df['line_basket_value_share'] = ratio(df.line_gross_candidate, df.basket_gross_candidate)
    df['line_basket_units_share'] = ratio(df.quantity, df.basket_units)
    df['relative_line_position'] = ratio(df.line_no, df.basket_lines)
    for c in ['unit_price', 'quantity', 'order_value', 'freight_cost', 'basket_gross_candidate', 'basket_units', 'product_volume_cm3']:
        df['log1p_' + c] = np.log1p(df[c].where(df[c] >= 0))
    df = df.drop(columns=['order_ts', 'coupon_code', 'store_channel'])
    df['banner_id'] = df.banner_id.astype('string')
    for c in df.select_dtypes(include='number'):
        if c != 'returned':
            df[c] = df[c].astype('float32')
    return df

def feature_columns(frame, include_promo=False):
    excluded = {'returned', 'order_line_id', 'order_id', '_order_time', '_month'}
    if not include_promo:
        excluded.add('promo_channel_code')
    return [c for c in frame if c not in excluded]

def feature_manifest(frame, include_promo=False):
    """Describe feature provenance for reproducible local experiments."""
    excluded = {'returned', 'order_line_id', 'order_id', '_order_time', '_month'}
    if not include_promo:
        excluded.add('promo_channel_code')
    rows = []
    for name in frame.columns:
        if name in excluded:
            continue
        source = 'derived'
        if name in {'customer_id', 'product_id', 'store_id'}:
            source = 'dimension key'
        elif name in {'channel', 'payment_method', 'ship_mode', 'unit_price', 'quantity',
                      'discount', 'freight_cost', 'order_value', 'line_no', 'items_in_order',
                      'gift_wrap', 'promo_channel_code', 'return_window_days', 'warehouse_zone',
                      'picker_shift', 'banner_id', 'session_clicks', 'device_type', 'tz_hint'}:
            source = 'transaction'
        elif name.endswith('_past_rate') or name.endswith('_past_count'):
            source = 'past labels; prior months only'
        rows.append({
            'feature': name,
            'source': source,
            'availability': 'before or at order time',
            'uses_labels': name.endswith('_past_rate') or name.endswith('_past_count'),
            'missing_behavior': 'native missing value' if frame[name].isna().any() else 'not observed',
        })
    return rows

def encode_categories(train, other, columns):
    """Fit vocabulary only on fitting rows; unseen categories become missing."""
    x = train[columns].copy()
    z = other[columns].copy()
    levels = {}
    for c in columns:
        if not pd.api.types.is_numeric_dtype(x[c]):
            values = sorted(x[c].dropna().astype(str).unique().tolist())
            levels[c] = values
            x[c] = pd.Categorical(x[c], categories=values)
            z[c] = pd.Categorical(z[c].where(z[c].isin(values)), categories=values)
    return x, z, levels


## 2. Load data and define forward evaluation

Orders are assigned to their latest timestamp. Each forecast spans six months. The 75-day delay applies to matured history; a separate full supervised-fit embargo is not claimed here. Selection uses earlier folds only.


In [ ]:
"""Embedded into the three standalone Kaggle challenger notebooks by build_challenger_notebooks.py."""
import gc, hashlib, json, os, platform, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

SEED = 42
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
NEEDED = {'train.csv','test.csv','customers.csv','products.csv','stores.csv','calendar.csv','sample_submission.csv'}
roots = [p.parent for p in Path('/kaggle/input').rglob('train.csv') if NEEDED.issubset(q.name for q in p.parent.iterdir())] if Path('/kaggle/input').exists() else []
local = Path('nexus-opti-ml-2026-operation-insight')
if local.exists() and NEEDED.issubset(q.name for q in local.iterdir()): roots.append(local)
assert len(roots) == 1, f'Attach one NEXUS dataset; found {roots}'
DATA = roots[0]
sample = pd.read_csv(DATA/'sample_submission.csv')
raw_train = pd.read_csv(DATA/'train.csv', low_memory=False)
raw_test = pd.read_csv(DATA/'test.csv', low_memory=False)
assert raw_train.order_line_id.is_unique and raw_test.order_line_id.is_unique
assert set(raw_train.order_id).isdisjoint(raw_test.order_id)
assert sample.columns.tolist() == ['order_line_id','returned']
assert set(sample.order_line_id) == set(raw_test.order_line_id)
dims = dimensions(DATA)
train = build_features(raw_train, dims)
test = build_features(raw_test, dims)
# Conservative split convention: assign an order to its latest line timestamp.
for frame, raw in ((train,raw_train),(test,raw_test)):
    frame['_order_time'] = pd.to_datetime(raw.order_ts).groupby(raw.order_id).transform('max').to_numpy()
    frame['_month'] = frame._order_time.dt.strftime('%Y-%m')
    assert frame.order_id.groupby(frame.order_id).transform('size').eq(frame.basket_lines).all()
del raw_train, raw_test, dims
gc.collect()
assert len(train) == 1400005 and len(test) == 479995

BASE_DROP = {'returned','customer_id','product_id','store_id','product_supplier_code','banner_id',
             'promo_channel_code','cpi','fuel_price','temperature_f','unemployment',
             'fuel_surcharge','customer_loyalty_tier','product_review_count',
             'store_seller_rating','store_manager_tenure_years',
             # These columns are created only for the risk-adjusted history
             # calculation. residual_oof contains the row's own label.
             'expected_oof','residual_oof'}
CALENDAR = {'month_sin','month_cos','weekday_sin','weekday_cos','holiday_flag'}
def add_calendar(frame):
    out = frame.copy()
    dt = out._order_time
    out['month_sin'] = np.sin(2*np.pi*(dt.dt.month-1)/12).astype('float32')
    out['month_cos'] = np.cos(2*np.pi*(dt.dt.month-1)/12).astype('float32')
    out['weekday_sin'] = np.sin(2*np.pi*dt.dt.dayofweek/7).astype('float32')
    out['weekday_cos'] = np.cos(2*np.pi*dt.dt.dayofweek/7).astype('float32')
    out['holiday_flag'] = pd.to_numeric(out.holiday_flag, errors='coerce').fillna(0).astype('float32') if 'holiday_flag' in out else 0.0
    return out
train, test = add_calendar(train), add_calendar(test)

def features(frame, seasonal=True):
    cols = [c for c in feature_columns(frame, include_promo=False)
            if c not in BASE_DROP and not c.endswith('_age_days') and c not in {'customer_tenure_days','store_age_days'}]
    if not seasonal: cols = [c for c in cols if c not in CALENDAR]
    assert not {'returned','expected_oof','residual_oof'} & set(cols), 'Target-derived column entered model features'
    return cols

def matrix(fit, other, cols):
    x, z, levels = encode_categories(fit, other, cols)
    cats = list(levels)
    for c in cats:
        x[c] = x[c].astype('string').fillna('__MISSING__').astype(str)
        z[c] = z[c].astype('string').fillna('__MISSING__').astype(str)
    for c in set(cols)-set(cats):
        x[c] = pd.to_numeric(x[c],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
        z[c] = pd.to_numeric(z[c],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
    return x, z, cats

def anchor(fit, other, iterations=1800, seed=SEED, extra_cols=()):
    cols = features(fit) + list(extra_cols)
    assert not {'returned','expected_oof','residual_oof'} & set(cols)
    assert set(cols) <= set(other.columns), f'Inference columns missing: {set(cols)-set(other.columns)}'
    x,z,cats = matrix(fit,other,cols)
    m = CatBoostClassifier(iterations=iterations, depth=5, learning_rate=.05,
          l2_leaf_reg=5, max_ctr_complexity=1, loss_function='Logloss',
          task_type='GPU', devices='0', random_seed=seed, verbose=False,
          allow_writing_files=False)
    m.fit(Pool(x,label=fit.returned,cat_features=cats))
    p = m.predict_proba(Pool(z,cat_features=cats))[:,1].astype('float32')
    return p,m,cols

def folds():
    for origin in ('2023-07-01','2024-01-01','2024-07-01'):
        start = pd.Timestamp(origin); end = start + pd.DateOffset(months=6)
        fit = train.loc[train._order_time < start].copy()
        valid = train.loc[(train._order_time >= start)&(train._order_time < end)].copy()
        if not len(valid): continue
        assert fit._order_time.max() < valid._order_time.min()
        assert set(fit.order_id).isdisjoint(valid.order_id)
        yield origin,fit,valid

def safe_auc(y,p):
    y=np.asarray(y); p=np.asarray(p)
    return float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None

def diagnostics(valid, pred):
    y=valid.returned.to_numpy(); p=np.asarray(pred)
    out={'pooled':safe_auc(y,p)}
    age=(valid._order_time.dt.year-valid._order_time.min().year)*12+valid._order_time.dt.month-valid._order_time.min().month
    groups={'early_horizon':age<3,'late_horizon':age>=3,'large_basket':valid.basket_lines>=51}
    for c in ('channel',):
        if c in valid:
            groups.update({f'{c}:{v}':valid[c].eq(v) for v in valid[c].dropna().unique()})
    for month in sorted(valid._month.unique()): groups[f'month:{month}']=valid._month.eq(month)
    for name,mask in groups.items():
        a=np.asarray(mask,dtype=bool); out[name]={'rows':int(a.sum()),'auc':safe_auc(y[a],p[a])}
    return out

def paired_order_ci(valid, a, b, n=200, seed=SEED):
    rng=np.random.default_rng(seed); ids=valid.order_id.to_numpy()
    unique,inv=np.unique(ids,return_inverse=True)
    gains=[]; y=valid.returned.to_numpy()
    for _ in range(n):
        counts=np.bincount(rng.integers(len(unique),size=len(unique)),minlength=len(unique))
        weights=counts[inv]
        if len(np.unique(y[weights>0]))==2:
            gains.append(float(roc_auc_score(y,b,sample_weight=weights)-
                               roc_auc_score(y,a,sample_weight=weights)))
    return np.quantile(gains,[.025,.5,.975]).tolist()

def save_prediction(name, frame, p):
    pd.DataFrame({'order_line_id':frame.order_line_id.to_numpy(),'returned':np.asarray(p)}).to_csv(OUT/name,index=False)

def submission(name,p):
    values=pd.DataFrame({'order_line_id':test.order_line_id,'returned':np.asarray(p,dtype='float64')})
    sub=sample[['order_line_id']].merge(values,on='order_line_id',how='left',sort=False,validate='one_to_one')
    assert len(sub)==479995 and sub.order_line_id.equals(sample.order_line_id)
    assert np.isfinite(sub.returned).all() and sub.returned.between(0,1).all()
    path=OUT/name; sub.to_csv(path,index=False)
    check=pd.read_csv(path)
    assert check.order_line_id.equals(sample.order_line_id)
    print('Saved',path,'SHA256',hashlib.sha256(path.read_bytes()).hexdigest())
    return path

def report(name, data):
    data['versions']={'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__}
    (OUT/name).write_text(json.dumps(data,indent=2,default=str))

print('Data',DATA,'Rows',len(train),len(test),'Output',OUT)


## 3. Train, compare, and write the submission

The notebook saves fold predictions, cohort diagnostics, an order-bootstrap comparison, package versions, and a checksummed submission CSV.


In [ ]:
"""Matched CUDA XGBoost constrained/unconstrained comparison."""
import xgboost as xgb

XGB_COLS=[c for c in features(train) if c not in {'month_of_year','day_of_week','hour','weekend'}]
STABLE=[c for c in XGB_COLS if c not in CALENDAR]
SEASON=[c for c in XGB_COLS if c in CALENDAR]
assert STABLE and SEASON and not set(STABLE)&set(SEASON)

def xgb_matrix(fit,other):
    a,b,_=encode_categories(fit,other,XGB_COLS)
    for c in XGB_COLS:
        if str(a[c].dtype)=='category':
            a[c]=a[c].cat.codes.replace(-1,np.nan).astype('float32')
            b[c]=b[c].cat.codes.replace(-1,np.nan).astype('float32')
        else:
            a[c]=pd.to_numeric(a[c],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
            b[c]=pd.to_numeric(b[c],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
    return a,b

CONSTRAINTS=json.dumps([STABLE,SEASON])
def xgb_fit(fit,other,constrained,depth=5,child=100,reg=40,rounds=1200):
    a,b=xgb_matrix(fit,other)
    dm=xgb.QuantileDMatrix(a,label=fit.returned.to_numpy(),max_bin=128)
    dv=xgb.DMatrix(b)
    params={'objective':'binary:logistic','eval_metric':'auc','tree_method':'hist',
        'device':'cuda','max_bin':128,'max_depth':depth,'min_child_weight':child,
        'lambda':reg,'eta':.04,'subsample':.9,'colsample_bytree':1.,'seed':SEED}
    if constrained: params['interaction_constraints']=CONSTRAINTS
    m=xgb.train(params,dm,num_boost_round=rounds,verbose_eval=False)
    pred=m.predict(dv).astype('float32')
    if constrained:
        # Verify every split path stays within its declared block.
        dump=m.get_dump(dump_format='json')
        def audit(node,used):
            if 'split' not in node: return
            name=node['split']; current=used|{name}
            assert current<=set(STABLE) or current<=set(SEASON), current
            for child_node in node['children']: audit(child_node,current)
        for tree in dump: audit(json.loads(tree),set())
    return pred,m

CONFIGS=[(3,50,20),(5,100,40),(5,200,80)]
fold_data=list(folds())
results={}; predictions={}; selection={}
for origin,fit,valid in fold_data:
    anchor_pred,_,_=anchor(fit,valid,1800)
    predictions[origin]={'anchor':anchor_pred}
    results[origin]={'anchor':diagnostics(valid,anchor_pred)}
    for depth,child,reg in CONFIGS:
        for constrained in (False,True):
            name=f'{"constrained" if constrained else "free"}_d{depth}_c{child}_r{reg}'
            p,_=xgb_fit(fit,valid,constrained,depth,child,reg)
            predictions[origin][name]=p
            results[origin][name]=diagnostics(valid,p)
            save_prediction(f'xgb_{origin[:7]}_{name}.csv',valid,p)
    save_prediction(f'xgb_{origin[:7]}_anchor.csv',valid,anchor_pred)
    print(origin,{k:round(v['pooled'],6) for k,v in results[origin].items()})
    del fit,valid
    gc.collect()

earlier=[k for k in results if k<'2024-07-01']
names=[n for n in results[earlier[0]] if n.startswith('constrained')]
def score(name):
    gains=[results[k][name]['pooled']-results[k]['anchor']['pooled'] for k in earlier]
    return (np.median(gains),min(gains))
best=max(names,key=score)
late_ok=all(results[k][best]['late_horizon']['auc'] is not None and
            results[k]['anchor']['late_horizon']['auc'] is not None and
            results[k][best]['late_horizon']['auc']-results[k]['anchor']['late_horizon']['auc']>=-.002
            for k in earlier)
use_xgb=score(best)[0]>=.001 and score(best)[1]>0 and late_ok
for k in results:
    results[k]['paired_order_ci']=paired_order_ci(
        next(v for o,_,v in fold_data if o==k),predictions[k]['anchor'],predictions[k][best],n=100)
print('Selected',best,'use constrained XGB',use_xgb)
anchor_test,anchor_model,anchor_cols=anchor(train,test,1800)
anchor_model.save_model(str(OUT/'constrained_xgb_anchor.cbm'))
if use_xgb:
    parts=best.split('_'); depth=int(parts[1][1:]); child=int(parts[2][1:]); reg=int(parts[3][1:])
    test_pred,model=xgb_fit(train,test,True,depth,child,reg)
    model.save_model(str(OUT/'constrained_xgb_model.json'))
else: test_pred=anchor_test
submission('submission_constrained_xgb.csv',test_pred)
report('constrained_xgb_report.json',{'folds':results,'selected':best,'use_xgb':use_xgb,
    'selection_rule':'earlier folds only; 2024-07 diagnostic','constraints':[STABLE,SEASON],
    'xgboost_version':xgb.__version__,'anchor_features':anchor_cols})
